In [15]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


df = pd.read_csv('../../../data/processed/review_histogram_v4.csv')
df['date_parsed'] = pd.to_datetime(df['date'], unit='s')

games = pd.read_csv('../../../data/raw/steam_stratified_sample_v4.csv')[['appid', 'name_store']]
df = df.merge(games, on='appid', how='left')
df['name_store'] = df['name_store'].fillna(df['appid'].astype(str))

print(df.shape)
print(df.dtypes)

(6003, 11)
appid                           int64
name                              str
stratum                           str
release_date                      str
date                            int64
recommendations_up              int64
recommendations_down            int64
rollup_type                       str
data_type                         str
date_parsed             datetime64[s]
name_store                        str
dtype: object


In [16]:
# 게임별 수집 현황
print(df.groupby(['stratum', 'appid', 'name'])['date'].count())

# 게임당 평균 월 수
rollups = df[df['data_type'] == 'rollups']
print(rollups.groupby('appid')['date'].count().describe())

stratum     appid    name                       
large_high  877200   Zero Caliber VR                120
            1049590  Eternal Return                  97
            1169040  Necesse                        107
            1272320  Diplomacy is Not an Option      81
            1340480  The Cosmic Wheel Sisterhood     63
                                                   ... 
small_high  2789810  Bingle Bingle                   56
            2868430  XiuzhenWorld                    59
            3087650  A Shelter Full of Cats         101
            3107900  Liminalcore                    107
            3413590  Raid Auctus                     66
Name: date, Length: 74, dtype: int64
count     74.000000
mean      51.121622
std       22.656429
min       26.000000
25%       32.250000
50%       40.500000
75%       69.750000
max      101.000000
Name: date, dtype: float64


In [17]:
df['release_date'] = pd.to_datetime(df['release_date'])
df['date_dt'] = pd.to_datetime(df['date'], unit='s')
df['months_since_release'] = (
    (df['date_dt'] - df['release_date']).dt.days / 30.44
).round(1)

In [18]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from dateutil.relativedelta import relativedelta

rollups = df[df['data_type'] == 'rollups'].copy()
rollups['month_int'] = rollups['months_since_release'].apply(lambda x: int(x // 1))
rollups = (
    rollups.groupby(['appid', 'name_store', 'stratum', 'month_int', 'release_date'], as_index=False)
           [['recommendations_up', 'recommendations_down']].sum()
           .sort_values(['appid', 'month_int'])
)

# 층별 대표 게임 2개씩 선택 (초기 3개월 리뷰 합산 기준 상위)
TARGET_STRATA = ['large_high', 'mid_high', 'small_high']
PICKS_PER_STRATUM = 2

early = rollups[rollups['month_int'] <= 3]
top_per_stratum = (
    early.groupby(['stratum', 'appid', 'name_store'])
         [['recommendations_up', 'recommendations_down']].sum()
         .assign(total=lambda x: x['recommendations_up'] + x['recommendations_down'])
         .reset_index()
         .sort_values(['stratum', 'total'], ascending=[True, False])
         .groupby('stratum')
         .head(PICKS_PER_STRATUM)
)
selected = top_per_stratum[top_per_stratum['stratum'].isin(TARGET_STRATA)]

# 층별 × 게임별 서브플롯
TICK_INTERVAL = 3  # 6개월마다 tick 표시
n_strata = len(TARGET_STRATA)
fig = make_subplots(
    rows=n_strata, cols=PICKS_PER_STRATUM,
    subplot_titles=[
        f"[{s}] {selected[selected['stratum']==s]['name_store'].iloc[i]}"
        for s in TARGET_STRATA
        for i in range(PICKS_PER_STRATUM)
        if i < len(selected[selected['stratum']==s])
    ],
    shared_xaxes=False, shared_yaxes=False,
    vertical_spacing=0.1, horizontal_spacing=0.08,
    x_title='출시 후 (월)',
)

for row_i, stratum in enumerate(TARGET_STRATA):
    picks = selected[selected['stratum'] == stratum]['appid'].tolist()
    for col_i, appid in enumerate(picks[:PICKS_PER_STRATUM]):
        game = rollups[rollups['appid'] == appid]
        show_legend = (row_i == 0 and col_i == 0)
        release_date = game['release_date'].iloc[0]
        all_months = sorted(game['month_int'].unique())

        # 6개월 간격 tick (0월은 항상 포함)
        tickvals = [m for m in all_months if m % TICK_INTERVAL == 0]
        ticktext = [
            f"{m}월<br>({(release_date + relativedelta(months=m)).year}년)"
            for m in tickvals
        ]

        fig.add_trace(go.Bar(
            x=game['month_int'], y=game['recommendations_up'],
            name='긍정', marker_color='steelblue',
            legendgroup='긍정', showlegend=show_legend,
        ), row=row_i+1, col=col_i+1)

        fig.add_trace(go.Bar(
            x=game['month_int'], y=-game['recommendations_down'],
            name='부정', marker_color='salmon',
            legendgroup='부정', showlegend=show_legend,
        ), row=row_i+1, col=col_i+1)

        fig.add_vline(
            x=0, line_dash='dash', line_color='gray', line_width=1,
            annotation_text='출시일', annotation_position='top right',
            annotation_font=dict(color='gray', size=10),
            row=row_i+1, col=col_i+1,
        )

        axis_key = 'xaxis' if (row_i == 0 and col_i == 0) else f'xaxis{row_i * PICKS_PER_STRATUM + col_i + 1}'
        fig.layout[axis_key].update(tickvals=tickvals, ticktext=ticktext, tickangle=0)

fig.update_layout(
    height=380 * n_strata, width=1100,
    title_text='층별 월별 리뷰 추이 (초기 3개월 상위 게임)',
    barmode='relative',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()